# 04 — Validate the point-in-time 13F panel

This notebook builds the first complete reporting-quarter snapshot from the local 13F mirror. It applies filing events in availability order, keeps future amendments out of earlier states, distinguishes supplements from restatements, and quarantines ambiguous records.

The notebook does **not** use the present-day issuer mapping, declare every cash-share line to be common stock, or construct an embedding matrix. Executed outputs and raw identifiers remain local.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_spine_13f import classify_instrument_rows, quarter_coverage_audit
from src.full_history_13f import read_parquet_columns
from src.point_in_time_13f import (
    FILING_EVENT_COLUMNS,
    POINT_IN_TIME_CONTRACT_VERSION,
    build_local_quarter_panel,
    eligible_holdings,
)

PRODUCT_ROOT = PROJECT_ROOT / "data" / "full_history" / "13f" / "v1"
TABLES_ROOT = PRODUCT_ROOT / "tables"
PRODUCT_ROOT.is_dir(), POINT_IN_TIME_CONTRACT_VERSION

## 1. Select the first decision-complete quarter

The date is derived from the local product rather than hard-coded. A quarter is eligible only when observation began by quarter-end and the normal filing-deadline cutoff is inside the data freeze.

In [ ]:
filings = read_parquet_columns(TABLES_ROOT / "filings", FILING_EVENT_COLUMNS)
coverage = quarter_coverage_audit(filings)
eligible_quarters = coverage.loc[coverage["eligible_complete_window"]].copy()
if eligible_quarters.empty:
    raise RuntimeError("No decision-complete quarter is available")

selected_quarter = eligible_quarters.iloc[0]
REPORT_DATE = selected_quarter["report_date_utc"]
DECISION_AT = selected_quarter["information_cutoff_utc"]

display(
    pd.DataFrame(
        {
            "selection": ["first decision-complete quarter"],
            "report_date": [REPORT_DATE],
            "information_cutoff": [DECISION_AT],
        }
    )
)

## 2. Build the canonical as-of state

Only filed-month partitions needed by visible self-reported holdings events are streamed. The selected columns preserve filing-line provenance and the identifiers needed for the next security-master review.

In [ ]:
HOLDING_COLUMNS = (
    "holding_id",
    "accession_no",
    "manager_cik",
    "period_of_report",
    "filed_at",
    "availability_timestamp_utc",
    "cusip",
    "title_of_class",
    "put_call",
    "shares_or_principal_type",
    "value_usd",
    "shares_or_principal_amount",
    "reported_issuer_cik",
)

panel = build_local_quarter_panel(
    PRODUCT_ROOT,
    report_date=REPORT_DATE,
    decision_at=DECISION_AT,
    holdings_columns=HOLDING_COLUMNS,
)
raw_holdings_asof_local = panel.holdings_asof
model_eligible_holdings_local = eligible_holdings(panel)

`raw_holdings_asof_local` is the complete local as-of holding-line DataFrame and can be opened in VS Code Data Wrangler. Do not display it in a version intended for GitHub. `model_eligible_holdings_local` removes incomplete or quarantined manager-period states but does not yet impose an equity universe.

## 3. Event ledger

Every filing remains accounted for: applied, future, excluded notice/combination, or quarantined.

In [ ]:
event_summary = (
    panel.event_ledger.groupby(
        ["event_status", "event_role", "coverage_state"],
        dropna=False,
    )
    .agg(filings=("accession_no", "size"), holding_rows_loaded=("holding_rows_loaded", "sum"))
    .reset_index()
    .sort_values(["event_status", "event_role", "coverage_state"])
)
display(event_summary)

## 4. Amendment transitions

An original initializes a state, `NEW_HOLDINGS` supplements it, and `RESTATEMENT` replaces it. An amendment without an observable original may be retained for audit but cannot become model-eligible.

In [ ]:
transition_summary = (
    panel.event_ledger.loc[panel.event_ledger["applied"]]
    .groupby(["event_role", "transition", "state_complete_after_event"], dropna=False)
    .agg(filings=("accession_no", "size"), holding_rows_loaded=("holding_rows_loaded", "sum"))
    .reset_index()
    .sort_values(["event_role", "transition"])
)
display(transition_summary)

## 5. Manager-period states

Model eligibility requires an applied, complete state and no visible quarantined event for that manager-period.

In [ ]:
state_summary = (
    panel.manager_periods.groupby(["state_status", "model_eligible"], dropna=False)
    .agg(
        manager_periods=("manager_cik", "size"),
        state_rows=("state_rows", "sum"),
        applied_events=("applied_events", "sum"),
        quarantined_events=("quarantined_events", "sum"),
    )
    .reset_index()
    .sort_values(["model_eligible", "state_status"], ascending=[False, True])
)
display(state_summary)

## 6. No-lookahead and reconciliation checks

These are blocking assertions. The unit tests separately prove that adding future amendments leaves an earlier snapshot unchanged.

In [ ]:
checks = pd.DataFrame(
    [
        {
            "check": "snapshot source events are available by decision",
            "violations": int(
                raw_holdings_asof_local["source_event_available_at_utc"].gt(panel.decision_at_utc).sum()
            ),
        },
        {
            "check": "applied ledger events are available by decision",
            "violations": int(
                panel.event_ledger.loc[panel.event_ledger["applied"], "available_at_utc"]
                .gt(panel.decision_at_utc)
                .sum()
            ),
        },
        {
            "check": "snapshot holding IDs are unique",
            "violations": int(raw_holdings_asof_local["holding_id"].duplicated().sum()),
        },
        {
            "check": "eligible rows come only from eligible states",
            "violations": int(
                (~model_eligible_holdings_local["state_model_eligible"].astype(bool)).sum()
            ),
        },
    ]
)
assert checks["violations"].eq(0).all(), checks
display(checks)

## 7. Provisional instrument channels

This is an audit classification, not the final common-equity filter. Options, debt-like instruments, funds, and other types remain separate.

In [ ]:
classified = classify_instrument_rows(raw_holdings_asof_local)
classified["value_usd_numeric"] = pd.to_numeric(classified["value_usd"], errors="coerce")
instrument_summary = (
    classified.groupby(["instrument_class", "classification_reason"], dropna=False)
    .agg(
        holding_rows=("holding_id", "size"),
        positive_value_rows=("value_usd_numeric", lambda values: int(values.gt(0).sum())),
        reported_value_usd=("value_usd_numeric", "sum"),
    )
    .reset_index()
    .sort_values("holding_rows", ascending=False)
)
display(instrument_summary)

## 8. Identifier coverage without present-day remapping

The snapshot preserves identifiers reported with the filing. Current resolved issuer mappings are intentionally excluded until a historically valid mapping policy exists.

In [ ]:
identifier_records = []
for column in ("cusip", "reported_issuer_cik"):
    values = raw_holdings_asof_local[column].astype("string").str.strip()
    populated = values.notna() & values.ne("")
    identifier_records.append(
        {
            "identifier": column,
            "holding_rows": len(values),
            "populated_rows": int(populated.sum()),
            "coverage_pct": 100 * float(populated.mean()),
            "distinct_populated": int(values[populated].nunique()),
        }
    )
identifier_summary = pd.DataFrame.from_records(identifier_records)
display(identifier_summary)

## 9. Decision gate

This milestone establishes a reproducible filing-event ledger and manager-period state at a historical information cutoff. The next gate is the point-in-time security master: determine which reported security lines are genuinely common equity and which issuer mappings were knowable at each date. Only then should we create the quarterly manager–asset matrix and fit the first simple embedding baseline.

The executed notebook belongs under ignored local `data/exploration/`; the committed source remains output-free.